# Code - Automated Pull Request Review Agent

In this exercise, we build an agent that reviews open Pull Requests on a GitHub repository, posts actionable review comments, and, when applicable, proposes fixes through a follow-up Pull Request.

We use the official [GitHub MCP Server](https://github.com/github/github-mcp-server) (remote hosted) so the agent can interact with GitHub through the same high-quality tools that GitHub Copilot uses.

## Environment Variables & Imports

Before using this notebook, please ensure you have obtained an API Key from your LLM backend and update the `.env` file to include it as follows:

```bash
GOOGLE_API_KEY=<copy API key here>
OPENAI_API_KEY=<copy API key here>
ANTRHOPIC_API_KEY=<copy API key here>
LLAMA_CPP_API_KEY=<copy API key here>
```


Additionally, this notebook expects a `GITHUB_PERSONAL_ACCESS_TOKEN` entry in `.env` with `repo` scope.

In [ ]:
import initialize_notebook # noqa

In [ ]:
import json
import os
import pathlib

import jinja2

from hslu.dlm03.common import backend as backend_lib
from hslu.dlm03.common import chat as chat_lib
from hslu.dlm03.common import presets
from hslu.dlm03.common.displays import ipython_display
from hslu.dlm03.common import tools as tools_lib
from hslu.dlm03.util import file as file_lib

## Parameters

In [ ]:
backend_name = "Gemma 4 26B"
backend_config = presets.CONFIGS_BY_NAME[backend_name]
backend = backend_lib.LLM.from_config(backend_config)

In [ ]:
# Remote GitHub MCP server - https://github.com/github/github-mcp-server
github_mcp_url = "https://api.githubcopilot.com/mcp/"
github_token = os.environ["GITHUB_PAT"]

# Target repository & pull request to review.
repo_owner = "VincentCoriou"
repo_name = "Bank-Account"
pr_numbers = [1]

# Agentic Harness

In [ ]:
import re
def remove_thoughts(text: str) -> str:
    # Remove well-formed <thought>...</thought> blocks (multiline safe)
    cleaned = re.sub(r"<thought>.*?</thought>", "", text, flags=re.DOTALL | re.IGNORECASE)

    # Optionally handle unclosed <thought> tags (strip until end)
    cleaned = re.sub(r"<thought>.*$", "", cleaned, flags=re.DOTALL | re.IGNORECASE)

    return cleaned.strip()

We list the tools exposed by the GitHub MCP server. The official server exposes a rich set of tools for interacting with pull requests, issues, files, branches, etc.

In [ ]:
async with tools_lib.mcp_session(github_mcp_url, authorization=github_token) as session:
    mcp_tools = await session.list_tools()
    tools = [tools_lib.tool_from_mcp(tool) for tool in mcp_tools.tools]
print(f"{len(tools)} tools exposed by the GitHub MCP server.")

In [ ]:
for tool in tools:
    print(tool["function"]["name"])

In [ ]:
system_instructions_template_string = \
"""You are an expert Software Engineer acting as an automated Pull Request reviewer.

For each Pull Request you are asked to review, you should:
1. Understand the intent of the change (read the title, body, and diff).
2. Inspect the changed files in depth using the available GitHub tools?
3. Identify any real issues: correctness bugs, missing error handling, unclear naming,  security issues, missing tests, style inconsistencies, etc.
4. Add comments to the pull request using "pull_request_review_write", do not output it to the user.

Guidelines:
- Be specific, concise, and constructive. Quote the offending code when it helps.
- Do NOT approve a PR that has obvious issues.
- Never push directly to the PR branch - always go through a follow-up PR.
- Only touch files that are part of the PR unless you are adding a new test that covers a change.
"""
system_instructions_template = jinja2.Template(system_instructions_template_string, undefined=jinja2.StrictUndefined)

In [ ]:
user_message_template_string = \
"""Please review pull request #{{ pr_number }} on {{ repo_owner }}/{{ repo_name }}."""
user_message_template = jinja2.Template(user_message_template_string, undefined=jinja2.StrictUndefined)

In [ ]:
system_instructions = system_instructions_template.render()
for pr_number in pr_numbers:
    chat_display = ipython_display.IPythonChatDisplay()
    user_message = user_message_template.render(
        repo_owner=repo_owner, repo_name=repo_name, pr_number=pr_number,
    )
    chat_display.show()
    chat = chat_lib.Chat(
        messages=[
            {"role": "system", "content": system_instructions},
            {"role": "user", "content": user_message},
        ],
        observers=[chat_display],
    )
    async with tools_lib.mcp_session(github_mcp_url, authorization=github_token) as session:
        mcp_tools = await session.list_tools()
        tools = [tools_lib.tool_from_mcp(tool) for tool in mcp_tools.tools]
        done = False
        while not done:
            response = backend.generate(chat, tools=tools)
            done = True
            message = response.choices[0].message
            if message.content is not None:
                message.content = remove_thoughts(message.content)
            chat.append(message)
            for tool_call in message.tool_calls or ():
                done = False
                tool_name = tool_call.function.name
                arguments = json.loads(tool_call.function.arguments)
                tool_call_result = await session.call_tool(tool_name, arguments)
                for content in tool_call_result.content:
                    tool_call_result_content = tools_lib.tool_call_result_from_mcp(
                        tool_call.id,
                        content,
                    )
                    chat.append(tool_call_result_content)